# 🚀 Zalo AI DataMate: Huấn Luyện Mô Hình Siêu Tốc trên Google Colab với QLoRA & Unsloth

Notebook này hướng dẫn **Fine-tune (SFT)** mô hình ngôn ngữ lớn (**Qwen2.5-Coder-7B** hoặc **3B / 1.5B**) trên tập dữ liệu 3.000 câu hỏi có sẵn của cuộc thi **Zalo AI Challenge: DataMate**.

### ⚡ Điểm tối ưu cho Google Colab (GPU T4 miễn phí):
1. **Unsloth Engine:** Tăng tốc độ huấn luyện gấp **2x - 5x**, giảm **70% VRAM** so với Hugging Face thông thường.
2. **QLoRA 4-bit (NF4):** Chỉ tiêu tốn ~6 - 7GB VRAM, vừa vặn hoàn hảo trong card Tesla T4 15GB VRAM của Colab.
3. **Thời gian huấn luyện:** Toàn bộ 3.000 mẫu câu hỏi chỉ mất **~15 - 20 phút** cho 1 epoch!
4. **Xuất trực tiếp sang GGUF / Ollama:** Sau khi train xong, có thể xuất thẳng sang file GGUF để nạp vào Ollama chạy benchmark ngay lập tức.

> ⚠️ **LƯU Ý QUAN TRỌNG:** Vào menu **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU** trước khi bắt đầu!

--- 
## 1. Kiểm tra Môi trường GPU

In [ ]:
# Đảm bảo hiển thị GPU Tesla T4 (15GB VRAM)
!nvidia-smi

--- 
## 2. Thiết lập Mã nguồn & Dữ liệu Dự án trên Colab
Kéo thả file `Zalo-ai.zip` vào tab **Files (📁)** bên trái của Colab rồi chạy ô bên dưới:

In [ ]:
import os

# Kiểm tra và giải nén Zalo-ai.zip
if os.path.exists('/content/Zalo-ai.zip') and not os.path.exists('/content/Zalo-ai/src'):
    print("📦 Đang giải nén mã nguồn và dữ liệu Zalo-ai.zip...")
    !unzip -q -o /content/Zalo-ai.zip -d /content/Zalo-ai

if os.path.exists('/content/Zalo-ai'):
    %cd /content/Zalo-ai
    print("✅ Thư mục làm việc hiện tại:", os.getcwd())
else:
    print("💻 Đang chạy trên máy Local hoặc thư mục hiện tại:", os.getcwd())

--- 
## 3. Cài đặt Unsloth & Thư viện Train Tối Ưu Tốc Độ

In [ ]:
# Cài đặt Unsloth (Bản tối ưu hóa riêng cho Colab)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets duckdb pandas
print("✅ Đã cài đặt xong môi trường huấn luyện Unsloth!")

--- 
## 4. Chuẩn bị Dữ liệu Huấn Luyện (3.000 mẫu từ tập Train)
Tự động chuyển đổi `train/questions.jsonl` và `train/answers.jsonl` thành định dạng ChatML tiêu chuẩn.

In [ ]:
# Chạy script tạo file train/sft_dataset.jsonl
!python scripts/prepare_sft_dataset.py

# Kiểm tra xem trước dữ liệu mẫu
import json
with open('train/sft_dataset.jsonl', encoding='utf-8') as f:
    sample = json.loads(f.readline())
    print("\n--- MẪU DỮ LIỆU ĐẦU VÀO HUẤN LUYỆN ---")
    print(json.dumps(sample, ensure_ascii=False, indent=2))

--- 
## 5. Tải Model Nền & Cấu hình LoRA (4-bit QLoRA)

Bạn có thể chọn một trong các model sau:
- `unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit` (Độ chính xác cao nhất, VRAM ~7GB, train ~20-25 phút)
- `unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit` (Cân bằng tốt, train ~12 phút)
- `unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit` (Siêu nhẹ, train cực nhanh ~6-8 phút)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024       # Đủ dài cho câu hỏi + schema + intent JSON
load_in_4bit = True         # Ép 4-bit để tiết kiệm 70% VRAM
model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"

print(f"⏳ Đang nạp model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

# Cấu hình LoRA Adapter siêu tốc
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # LoRA rank (16 là tối ưu cho text classification & extraction)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,         # Unsloth tối ưu 0 dropout để chạy nhanh nhất
    bias="none",
    use_gradient_checkpointing="unsloth", # Giảm VRAM thêm 30%
    random_state=3407,
)
print("✅ Cấu hình LoRA thành công!")

--- 
## 6. Định dạng Prompt với Chat Template của Qwen2.5

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Thiết lập template Qwen-2.5 chuẩn
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Tải 3.000 mẫu train
dataset = load_dataset("json", data_files="train/sft_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"✅ Đã chuẩn bị xong {len(dataset)} mẫu huấn luyện!")

--- 
## 7. Khởi chạy Huấn Luyện (Training Execution)

Tham số được tinh chỉnh riêng cho **GPU Tesla T4** để đạt tốc độ tối đa:

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False, # Không pack để giữ nguyên cấu trúc đối thoại
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective Batch Size = 8
        warmup_steps=10,
        max_steps=200,                  # Huấn luyện 200 bước (~1 epoch trên 3k mẫu)
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="paged_adamw_8bit",       # Tiết kiệm bộ nhớ optimizer
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("🚀 BẮT ĐẦU HUẤN LUYỆN...")
trainer_stats = trainer.train()
print("🎉 HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")

--- 
## 8. Kiểm tra Dự đoán Trực tiếp Sau khi Huấn luyện (Inference Test)

In [ ]:
# Chuyển model sang chế độ Inference siêu tốc của Unsloth
FastLanguageModel.for_inference(model)

test_prompt = "thang 7/2026 co bao nhiu don co dthu tu 55.000 dong tro len (as_of_date: 2026-07-31)"

messages = [
    {"role": "system", "content": "Bạn là chuyên gia phân tích intent và điều phối dữ liệu cho hệ thống Zalo AI DataMate. Hãy xác định miền nghiệp vụ (domain), bảng đích (recommended_table), và định dạng đáp án (expected_answer_type) cho câu hỏi của chủ shop."},
    {"role": "user", "content": f'Câu hỏi: "{test_prompt}"'}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True)
decoded = tokenizer.batch_decode(outputs)
print("\n--- KẾT QUẢ DỰ ĐOÁN CỦA MODEL VỪA TRAIN ---")
print(decoded[0].split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip())

--- 
## 9. Lưu Mô Hình & Xuất File (GGUF / Ollama hoặc LoRA Adapter)

Bạn có 2 cách lưu:
- **Cách 1: Lưu LoRA Adapter** (Nhẹ ~50MB, tải về máy nhanh).
- **Cách 2: Xuất trực tiếp sang file GGUF 4-bit** để nạp thẳng vào Ollama chạy thi đấu!

In [ ]:
# Cách 1: Lưu LoRA weights (Nhẹ ~50MB)
os.makedirs("models/qwen_datamate_lora", exist_ok=True)
model.save_pretrained("models/qwen_datamate_lora")
tokenizer.save_pretrained("models/qwen_datamate_lora")
print("✅ Đã lưu LoRA adapter vào: models/qwen_datamate_lora")

# Cách 2 (Tùy chọn): Xuất sang file GGUF q4_k_m để dùng cho Ollama
# Bỏ ghi chú dòng dưới nếu bạn muốn xuất GGUF:
# model.save_pretrained_gguf("models/qwen_datamate_gguf", tokenizer, quantization_method="q4_k_m")